# Setup della sessione Colab – AML Project 2

<a href="https://colab.research.google.com/github/frastud812/aml-project-2/blob/main/notebooks/00_session_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Questo notebook prepara l'ambiente per lavorare con SplatPose su Colab. **Va eseguito all'inizio di ogni sessione**,
perché tutto ciò che sta in `/content` (codice, dati, librerie compilate) sparisce quando la sessione si chiude.

**Come si usa**
1. *Runtime → Cambia tipo di runtime → **T4 GPU***.
2. Modifica **solo** la cella di configurazione qui sotto.
3. *Runtime → Esegui tutto*. Alla fine l'ambiente è pronto: aggiungi le tue celle in fondo, o copia il notebook e
   continua lì.

**Cosa resta tra una sessione e l'altra:** solo ciò che sta su Google Drive (`aml-project-2/`) e su GitHub.

**Prima di salvarlo su GitHub:** *Modifica → Cancella tutti gli output*, e salvalo sul **tuo branch**, mai su `main`.

## Configurazione
L'unica cella da modificare. `EXP` diventa la cartella `runs/<EXP>/` su Drive: il nome deve dire con quali
impostazioni è stato prodotto il contenuto (vedi README), es. `repro_v210_it30k_k175`. Per seed diversi usate
nomi diversi (es. `..._s1`).

In [ ]:
AUTHOR = 'nome'                     # chi lancia gli esperimenti: finisce in info.json e metrics.csv
BRANCH = 'main'                     # il proprio branch di lavoro, es. 'p1/gs-training'
EXP    = 'repro_v210_it30k_k175'    # nome dell'esperimento -> cartella runs/<EXP>/ su Drive
SEED   = 0

PREPARE_DATA = False   # True solo la PRIMA volta, se data/ su Drive è vuota (sezione 5)
CACHE_WHEELS = True    # salva su Drive le estensioni CUDA compilate: dalla 2a sessione niente compilazione

REPO_URL = 'https://github.com/frastud812/aml-project-2.git'
DRIVE    = '/content/drive/MyDrive/aml-project-2'
WORK     = '/content/aml-project-2'

## 1. Funzioni di servizio e controllo della GPU

In [ ]:
import os, sys, json, csv, glob, shutil, subprocess, datetime
from pathlib import Path
from IPython import get_ipython

def sh(cmd):
    # Esegue un comando di shell mostrando l'output; si ferma se il comando fallisce.
    print(f'$ {cmd}')
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code:
        raise RuntimeError(f'Comando fallito (exit {code}): {cmd}')

import torch
print('Python', sys.version.split()[0], '| PyTorch', torch.__version__, '| CUDA', torch.version.cuda)
assert torch.cuda.is_available(), "Nessuna GPU: Runtime -> Cambia tipo di runtime -> T4 GPU, poi riesegui tutto."
print('GPU:', torch.cuda.get_device_name(0))

## 2. Google Drive
Monta Drive e controlla che la cartella condivisa sia raggiungibile. Se l'assert fallisce: su drive.google.com,
*Condivisi con me* → clic destro su `aml-project-2` → *Organizza → Aggiungi scorciatoia → Il mio Drive*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

assert os.path.isdir(DRIVE), f'{DRIVE} non trovata: aggiungi la scorciatoia a "Il mio Drive" (vedi README).'
for sub in ['data', 'runs', 'synthetic', 'heads', 'results', 'env']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)

DATA_ZIPS = ['MAD-Sim.zip', 'MAD-Sim_3dgs.zip', 'model.zip']
missing = [z for z in DATA_ZIPS if not os.path.exists(f'{DRIVE}/data/{z}')]
print('Dati su Drive:', 'tutti presenti' if not missing else f'MANCANO {missing}')

## 3. Codice da GitHub
Clona il proprio branch (o lo aggiorna se è già stato clonato in questa sessione), con i submodule CUDA.
Il codice **non si modifica qui**: si corregge in locale, si fa push sul branch e qui si riesegue questa cella.

In [ ]:
if os.path.isdir(f'{WORK}/.git'):
    sh(f'git -C {WORK} fetch origin')
    sh(f'git -C {WORK} checkout {BRANCH}')
    sh(f'git -C {WORK} pull --ff-only origin {BRANCH}')
else:
    sh(f'git clone --recursive -b {BRANCH} {REPO_URL} {WORK}')
sh(f'git -C {WORK} submodule update --init --recursive')
os.chdir(WORK)

COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Branch {BRANCH} @ {COMMIT}')

## 4. Dipendenze ed estensioni CUDA
Il codice originale è stato scritto per Python 3.7 e PyTorch 1.12; Colab ha versioni molto più recenti. Per questo:
- i pacchetti si installano **senza** le versioni fissate di `requirements.txt` (non esistono per il Python di Colab);
- nei sorgenti CUDA mancano due `#include` che i compilatori recenti richiedono: li aggiungiamo qui;
- la compilazione usa `--no-build-isolation`, altrimenti pip compila in un ambiente separato senza `torch`;
- `TORCH_CUDA_ARCH_LIST=7.5` compila solo per la T4, molto più veloce;
- `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1`: dal PyTorch 2.6 `torch.load` rifiuta i checkpoint "vecchio stile" come quello
  del modello di retrieval.

Con `CACHE_WHEELS = True` le estensioni compilate vengono salvate in `env/wheels/` su Drive: nelle sessioni
successive si reinstallano in pochi secondi. Se Colab aggiorna PyTorch, vengono ricompilate da sole.

In [ ]:
sh('pip install -q plyfile easydict einops kornia yacs loguru wandb gdown')

def add_include(path, after, include):
    # Aggiunge `include` subito dopo la riga `after`, se non c'è già.
    p = Path(path)
    src = p.read_text()
    if include in src:
        return print(f'già corretto: {path}')
    assert after in src, f'"{after}" non trovato in {path}'
    p.write_text(src.replace(after, f'{after}\n{include}', 1))
    print(f'corretto: {path}')

add_include('submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h', '#include <iostream>', '#include <cstdint>')
add_include('submodules/simple-knn/simple_knn.cu', '#include <vector>', '#include <cfloat>')

In [ ]:
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

def sub_rev(name):
    return subprocess.check_output(['git', '-C', f'submodules/{name}', 'rev-parse', '--short', 'HEAD'], text=True).strip()

key = (f'torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}'
       f"_dgr{sub_rev('diff-gaussian-rasterization')}_knn{sub_rev('simple-knn')}")
WHEELS = f'{DRIVE}/env/wheels/{key}'
cached = sorted(glob.glob(f'{WHEELS}/*.whl'))

if CACHE_WHEELS and len(cached) == 2:
    print('Uso le estensioni già compilate in', WHEELS)
    sh('pip install -q --force-reinstall --no-deps ' + ' '.join(cached))
else:
    print('Compilo le estensioni CUDA (qualche minuto)...')
    shutil.rmtree('/content/wheels', ignore_errors=True)
    sh('pip wheel --no-build-isolation --no-deps -w /content/wheels '
       './submodules/diff-gaussian-rasterization ./submodules/simple-knn')
    built = sorted(glob.glob('/content/wheels/*.whl'))
    sh('pip install -q --force-reinstall --no-deps ' + ' '.join(built))
    if CACHE_WHEELS:
        os.makedirs(WHEELS, exist_ok=True)
        for w in built:
            shutil.copy(w, WHEELS)
        print('Salvate su Drive in', WHEELS)

sh("python -c \"import torch, diff_gaussian_rasterization, simple_knn._C; print('Estensioni OK, CUDA:', torch.cuda.is_available())\"")

## 5. Solo la prima volta: preparare i dati su Drive
**Una sola persona, una sola volta**, con `PREPARE_DATA = True`. Scarica MAD-Sim e il modello di retrieval dai link
ufficiali, esegue `prepare_lego_ad.py` (che sistema i nomi dei file e crea `MAD-Sim_3dgs`) e carica i tre zip in
`data/` su Drive. Dopo, `data/` è **in sola lettura**. Se gdown segnala troppi download, riprova più tardi o scarica a
mano dal [repository PAD](https://github.com/EricLee0224/PAD) e carica gli zip in `data/`.

In [ ]:
if not PREPARE_DATA:
    print('Saltato (PREPARE_DATA = False).')
elif not missing:
    print('I dati sono già su Drive: niente da preparare.')
else:
    PREP = '/content/prep'
    os.makedirs(PREP, exist_ok=True)
    sh(f'gdown 1XlW5v_PCXMH49RSKICkskKjd2A5YDMQq -O {PREP}/MAD-Sim_original.zip')
    sh(f'unzip -q -o {PREP}/MAD-Sim_original.zip -d {PREP}/unz')
    roots = [p.parent for p in Path(f'{PREP}/unz').rglob('01Gorilla') if p.is_dir() and '__MACOSX' not in p.parts]
    assert roots, 'Struttura dello zip inattesa: non trovo la cartella 01Gorilla'
    shutil.rmtree(f'{WORK}/MAD-Sim', ignore_errors=True)
    shutil.move(str(roots[0]), f'{WORK}/MAD-Sim')

    sh(f'cd {WORK} && python prepare_lego_ad.py')
    sh(f'cd {WORK} && zip -qr {PREP}/MAD-Sim.zip MAD-Sim && zip -qr {PREP}/MAD-Sim_3dgs.zip MAD-Sim_3dgs')
    sh(f'gdown 16FOwaqQE0NGY-1EpfoNlU0cGlHjATV0V -O {PREP}/model.zip')

    for z in DATA_ZIPS:
        shutil.copy(f'{PREP}/{z}', f'{DRIVE}/data/{z}')
        print('caricato su Drive:', z)
    missing = []

## 6. Dati in locale
Copia gli zip da Drive in `/content` e li decomprime dentro il repo, dove SplatPose li cerca (`MAD-Sim/`,
`MAD-Sim_3dgs/`, `PAD_utils/model/`). Leggere migliaia di immagini direttamente da Drive sarebbe lentissimo.

In [ ]:
assert not missing, f'Mancano su Drive {missing}: esegui la sezione 5 con PREPARE_DATA = True (una persona, una volta).'

def unzip_from_drive(name, dest, check):
    if os.path.exists(check):
        return print(f'{name}: già pronto')
    shutil.copy(f'{DRIVE}/data/{name}', f'/content/{name}')
    sh(f'unzip -q -o /content/{name} -d {dest}')
    assert os.path.exists(check), f'Dopo {name} non trovo {check}: controlla la struttura dello zip'
    print(f'{name}: ok')

unzip_from_drive('MAD-Sim.zip', WORK, f'{WORK}/MAD-Sim/01Gorilla')
unzip_from_drive('MAD-Sim_3dgs.zip', WORK, f'{WORK}/MAD-Sim_3dgs/01Gorilla')
unzip_from_drive('model.zip', f'{WORK}/PAD_utils', f'{WORK}/PAD_utils/model/indoor_ds_new.ckpt')

## 7. Risultati direttamente su Drive
SplatPose salva in `MAD-Sim_3dgs/results_<prefix>_<seed>/<classe>/`. Questa cella trasforma quella cartella in un
collegamento a `runs/<EXP>/` su Drive: ogni file salvato finisce subito su Drive, e una disconnessione non fa perdere
ciò che è già stato prodotto. Lanciate SplatPose con `-p {EXP} -seed {SEED}`.

In [ ]:
def link_experiment(exp=EXP, seed=SEED):
    target = f'{DRIVE}/runs/{exp}'
    link = f'{WORK}/MAD-Sim_3dgs/results_{exp}_{seed}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(link):
        os.remove(link)
    elif os.path.exists(link):
        raise RuntimeError(f'{link} esiste già come cartella locale: spostane il contenuto su Drive prima.')
    os.symlink(target, link)
    print(f'{link}\n  -> {target}')
    return target

RUN_DIR = link_experiment()

## 8. Registrare gli esperimenti
- `save_info(classe, **impostazioni)` scrive `runs/<EXP>/<classe>/info.json` con autore, data, branch e commit.
- `log_metrics(classe, ...)` aggiunge una riga a `results/metrics.csv` su Drive (il file condiviso delle metriche).

In [ ]:
def now():
    return datetime.datetime.now().isoformat(timespec='seconds')

def save_info(classname, **settings):
    d = f'{DRIVE}/runs/{EXP}/{classname}'
    os.makedirs(d, exist_ok=True)
    info = dict(author=AUTHOR, date=now(), branch=BRANCH, commit=COMMIT, exp=EXP, seed=SEED,
                classname=classname, **settings)
    with open(f'{d}/info.json', 'w') as f:
        json.dump(info, f, indent=2)
    print('salvato', f'{d}/info.json')

METRICS_FIELDS = ['date', 'author', 'exp', 'classname', 'seed', 'commit',
                  'image_auroc', 'pixel_auroc', 'aupro', 'notes']

def log_metrics(classname, image_auroc=None, pixel_auroc=None, aupro=None, notes=''):
    path = f'{DRIVE}/results/metrics.csv'
    new = not os.path.exists(path)
    row = dict(date=now(), author=AUTHOR, exp=EXP, classname=classname, seed=SEED, commit=COMMIT,
               image_auroc=image_auroc, pixel_auroc=pixel_auroc, aupro=aupro, notes=notes)
    with open(path, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=METRICS_FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    print('aggiunta riga a', path)

print('Ambiente pronto. Branch', BRANCH, '@', COMMIT, '| esperimento', EXP)

## 9. Da qui in poi: il proprio lavoro
Esempio di uso (lasciato commentato). Una classe alla volta; le opzioni di SplatPose si vedono con
`!python train_and_render.py --help` (es. `-gauss_iters`, `-k` per i passi di raffinamento della posa).

In [ ]:
# CLASS = '02Unicorn'
# save_info(CLASS, gauss_iters=30000, pose_steps=175)
# !python train_and_render.py -c {CLASS} -p {EXP} -seed {SEED}
# log_metrics(CLASS, image_auroc=..., pixel_auroc=..., aupro=..., notes='')

## Fine sessione
Forza la scrittura su Drive di tutto ciò che è stato salvato. Eseguila prima di chiudere.

In [ ]:
# drive.flush_and_unmount()